<a href="https://colab.research.google.com/github/mdudavillela/gradient-convergence-mlp/blob/main/notebooks/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Main Notebook

Este notebook dita o projeto: ele não tem a lógica pesada em si, só chama funções que moram nos módulos dentro de `src/` (mesmo papel que um `main.R` com `source()` cumpriria se fosse programado em R).

**Como funciona esse notebook?**
```
main.ipynb -> clona o repositório -> importa coleta.py -> pega a API key -> roda coletar_dataset() -> gera data/tmdb_raw.csv
```

Por enquanto ele cobre só o "pré-processamento": importar/coletar o dataset. Os próximos módulos entram depois, cada um importado aqui quando chegar a vez.

## 1. Clonar o repositório

O colab roda numa máquina virtual temporária, ela não tem acesso ao repositório do github por padrão, então o primeiro passo é baixar uma cópia dele pra dentro da máquina (`src/coleta.py`).

- `!git clone {REPO_URL}` — o `!` diz ao notebook "isso não é python, é um comando de terminal". É o mesmo `git clone` que seria usado fora do colab.
- `%cd {REPO_DIR}` — o `%` marca um comando "mágico" do Jupyter. Esse em específico muda a pasta de trabalho pra dentro do repositório clonado, pra que os caminhos relativos usados nas próximas células (tipo `"src"` ou `"data/tmdb_raw.csv"`) funcionem certinho.

In [5]:
import os  # biblioteca pra interagir com o sistema operacional

REPO_URL = "https://github.com/mdudavillela/gradient-convergence-mlp.git"
REPO_DIR = "gradient-convergence-mlp"  # nome da pasta que o clone cria (sempre o nome do repositório)

# só clona se a pasta ainda não existir, evita erro caso essa célula rode duas vezes na mesma sessão
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"Pasta '{REPO_DIR}' já existe - pulando o clone.")

%cd {REPO_DIR}

Cloning into 'gradient-convergence-mlp'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 39 (delta 6), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 21.67 KiB | 7.22 MiB/s, done.
Resolving deltas: 100% (6/6), done.
/content/gradient-convergence-mlp/gradient-convergence-mlp


## 2. Importar o módulo de coleta

`sys.path` é a lista de pastas onde o python procura por módulos quando se escreve `import algo`. Por padrão, `src/` não está nessa lista. Sem esse passo, `from coleta import coletar_dataset` daria `ModuleNotFoundError`.

In [6]:
import sys
sys.path.append("src")

from coleta import coletar_dataset

print("Módulo 'coleta' importado com sucesso.")

Módulo 'coleta' importado com sucesso.


## 3. Configurar a API key

Tenta carregar a chave dos **Secrets** do colab primeiro (configurar uma vez: ícone 🔑 na barra lateral → "Add new secret" → nome `TMDB_API_KEY`).

In [7]:
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("TMDB_API_KEY")
    print("API key carregada dos Secrets do colab.")
except Exception:
    api_key = getpass.getpass("Cole sua API Key (v3 auth) do TMDB: ")

API key carregada dos Secrets do colab.


## 4. Rodar a coleta

Aqui chamamos a função "chefe" do `coleta.py`. O notebook não precisa saber como a coleta funciona por dentro, só precisa saber o nome da função, quais parâmetros ela espera, e o que ela devolve (um `DataFrame`). Essa separação entre "o que fazer" (aqui) e "como fazer" (dentro do módulo) é o ganho principal de organizar o código assim.

Na primeira vez, demora alguns minutos (milhares de requisições à API). Depois disso, mesmo numa sessão nova do colab, se `data/tmdb_raw.csv` já estiver commitado no repositório, essa célula carrega ele na hora em vez de recoletar (é a lógica de cache de dentro da `coletar_dataset`).


In [9]:
df_raw = coletar_dataset(
    api_key = api_key,
    cache_path = "data/tmdb_raw.csv",
    n_paginas = 250,          # ~20 filmes por página -> até ~5000 filmes candidatos
    qtd_votos_min = 100,      # mínimo de votos para a nota do público ser confiável
    numero_threads = 10,      # requisições simultâneas na busca de detalhes
    atualizar_do_zero = False, # mudar para True só se quiser recoletar dados atualizados -> NUNCA DURANTE A IC
)

print(df_raw.shape)
df_raw.head()

Etapa 1a: descobrindo IDs de filmes candidatos...
  página 50/250 | 1000 ids coletados até agora
  página 100/250 | 2000 ids coletados até agora
  página 150/250 | 3000 ids coletados até agora
  página 200/250 | 4000 ids coletados até agora
  página 250/250 | 5000 ids coletados até agora
Total de IDs candidatos: 4769

Etapa 1b: buscando detalhes de cada filme (em paralelo)...
  500/4769 filmes processados...
  1000/4769 filmes processados...
  1500/4769 filmes processados...
  2000/4769 filmes processados...
  2500/4769 filmes processados...
  3000/4769 filmes processados...
  3500/4769 filmes processados...
  4000/4769 filmes processados...
  4500/4769 filmes processados...

Coleta concluída e salva em 'data/tmdb_raw.csv'. Shape: (4766, 12)
(4766, 12)


,id,title,budget,revenue,runtime,popularity,vote_average,vote_count,release_date,original_language,genres,production_companies_count
0,1368337,The Odyssey,250000000,1751248000,173,298.6091,8.012,3996,2026-07-15,en,"[Adventure, Action, Fantasy]",2
1,1423191,Resident Evil,75000000,212394895,95,433.5555,7.308,694,2026-09-16,en,"[Horror, Science Fiction, Adventure]",7
2,1375646,Colony,0,0,123,172.1553,8.109,1014,2026-05-21,ko,"[Horror, Action, Science Fiction]",3
3,1204680,Coyote vs. Acme,72000000,88043208,103,290.8185,7.525,632,2026-08-17,en,"[Comedy, Adventure, Family]",4
4,1032863,The Love Hypothesis,0,0,112,205.4903,8.039,441,2026-09-23,en,"[Romance, Comedy]",3


## 5. Próximo passo: levar o CSV para o repositório

Se a coleta acabou de rodar pela primeira vez, `data/tmdb_raw.csv` existe só nesta máquina temporária do colab e assim que a sessão desconectar, ele some, porque ainda não está no github.

Jeito mais simples: baixar o arquivo do colab (ícone de pasta 📁 na lateral → botão direito em `data/tmdb_raw.csv` → Download) e adicionar ele ao repositório do mesmo jeito que foi feito com o `coleta.py` (Add file → Create new file ou Upload files → commit).

Depois disso, qualquer sessão futura (minha ou de outra pessoa) já encontra o cache pronto sem precisar de API key nem esperar os minutos de coleta de novo.

## 6. Importar o módulo de pré-processamento


In [29]:
!git pull
print(os.listdir("src"))

Updating 5f23917..309a93d
error: The following untracked working tree files would be overwritten by merge:
	data/tmdb_raw.csv
Please move or remove them before you merge.
Aborting
['coleta.py']


In [32]:
# Força o descarte de alterações locais conflitantes e atualiza o repositório
!git reset --hard origin/main
!git pull

import sys
import os

# Encontra a raiz do repositório de forma robusta para evitar problemas de caminhos duplicados/aninhados
caminho_atual = os.getcwd()
if "gradient-convergence-mlp" in caminho_atual:
    raiz_repo = caminho_atual.split("gradient-convergence-mlp")[0] + "gradient-convergence-mlp"
    pasta_src = os.path.join(raiz_repo, "src")
    if pasta_src not in sys.path:
        sys.path.append(pasta_src)

from preprocessamento import (
    carregar_dados_validos,
    criar_alvo,
    selecionar_features,
    tratar_categoricas,
    dividir_e_normalizar,
)

print("Módulo 'preprocessamento' importado com sucesso.")

HEAD is now at 309a93d Update README.md
Already up to date.


ModuleNotFoundError: No module named 'preprocessamento'

## 7. Rodar o pré-processamento
Encadeando as funções na ordem do pipeline:

`tmdb_raw.csv -> carregar_dados_validos() -> criar_alvo() -> selecionar_features() ->
tratar_categoricas() -> dividir_e_normalizar() -> (x_treino, x_teste, y_treino, y_teste)`

In [ ]:
df = carregar_dados_validos("data/tmdb_raw.csv")
df = criar_alvo(df)

x_numerico = selecionar_features(df)
x_categorico = tratar_categoricas(df)
y = df["log_revenue"].values

x_treino, x_teste, y_treino, y_teste, nomes_colunas = dividir_e_normalizar(x_numerico, x_categorico, y)

print("Treino:", x_treino.shape)
print("Teste :", x_teste.shape)
print("Colunas:", nomes_colunas)